# 08. Manuscript Figures

[![Repo](https://img.shields.io/badge/GitHub-metaheuristic--budget--reproduction-181717?logo=github&logoColor=white)](https://github.com/prakash-ukhalkar/metaheuristic-budget-reproduction) [![License](https://img.shields.io/badge/license-MIT-green)](../LICENSE) [![Author](https://img.shields.io/badge/author-Prakash%20Ukhalkar-blue?logo=orcid&logoColor=white)](https://orcid.org/0000-0002-0452-6574) [![Python](https://img.shields.io/badge/python-3.10%2B-blue)](../requirements.txt)

**Source script:** `src/make_figures.py` &nbsp;|&nbsp; **Notebook 8 of 10**

Generates the final, manuscript-numbered figures 1-8 (PNG + PDF).

Part of *Budget-Controlled Reproduction Study of Nature-Inspired Metaheuristics* — a reproduction study comparing six metaphor-based metaheuristics (GWO, WOA, SCA, SSA, HHO, AOA) against five established baselines (DE, PSO, L-SHADE, CMA-ES, random search) on constrained engineering design problems, under matched evaluation budgets and tuning effort.

See the [repository README](../README.md) for installation and full reproduction instructions, and [notebooks/README.md](README.md) for the notebook index and suggested run order.

---


# Final Manuscript Figure Generation

![Python](https://img.shields.io/badge/python-3.10%2B-blue) ![Status](https://img.shields.io/badge/status-research--reproduction-lightgrey) ![License](https://img.shields.io/badge/license-MIT-green)

Generates the final, numbered set of eight manuscript figures (design schematic, critical-difference diagram, win/tie/loss heat map, ECDF, convergence panels, boxplot, tuning effect, constraint-scheme sensitivity) as both PNG and PDF.


## Imports

External libraries and project modules used by this notebook.

In [ ]:
import json
import os

import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

from analysis import (BASELINES, ORDER, TEST_GROUP, TRAIN_PROBLEMS,
                      load, pairwise_wilcoxon, friedman_nemenyi)
from problems import PROBLEM_MAP

FIG = os.path.join(os.path.dirname(__file__), "..", "figures")
os.makedirs(FIG, exist_ok=True)
BUDGET = 15000

plt.rcParams.update({
    "font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linewidth": 0.5,
    "figure.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.03,
    "font.family": "DejaVu Sans", "pdf.fonttype": 42,
})
MARK = {"GWO": "o", "WOA": "s", "SCA": "^", "SSA": "v", "HHO": "D", "AOA": "P",
        "DE": "X", "PSO": "*", "LSHADE": "<", "CMAES": ">", "RS": "h"}
DASH = {"GWO": "-", "WOA": "--", "SCA": "-.", "SSA": ":", "HHO": "-", "AOA": "--",
        "DE": "-.", "PSO": ":", "LSHADE": "-", "CMAES": "--", "RS": "-."}
COL = {a: ("black" if a in TEST_GROUP else "0.45") for a in ORDER}

### `save`

Writes a figure as both PNG and PDF into the figures directory.


In [ ]:
def save(fig, stem):
    fig.savefig(f"{FIG}/{stem}.png")
    fig.savefig(f"{FIG}/{stem}.pdf")
    plt.close(fig)

### `fig1_design`

Figure 1: study design schematic (problems, algorithms, constraint schemes, tuning, statistics).


In [ ]:
def fig1_design():
    fig, ax = plt.subplots(figsize=(7.0, 3.2))
    ax.axis("off"); ax.set_xlim(0, 10); ax.set_ylim(0, 5)

    def box(x, y, w, h, t):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.07",
                                    fc="white", ec="k", lw=0.9))
        ax.text(x + w / 2, y + h / 2, t, ha="center", va="center", fontsize=7.5)

    def arrow(x1, y1, x2, y2):
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="->",
                                     mutation_scale=8, lw=0.8, color="k"))

    box(0.1, 3.55, 2.2, 1.05, "9 design problems\nverified against\npublished optima")
    box(0.1, 2.00, 2.2, 1.05, "11 algorithms\n6 metaphor-based\n5 baselines")
    box(0.1, 0.45, 2.2, 1.05, "3 constraint\nschemes: Deb,\nstatic, $\\epsilon$")
    box(3.05, 2.70, 2.1, 1.05, "matched-budget\ntuning on 3\ntraining problems")
    box(3.05, 1.15, 2.1, 1.05, "author-default\nhyperparameters")
    box(5.95, 1.60, 1.9, 1.85, "25 seeded runs\n15,000 evaluations\nshared evaluator\nseeds paired\nacross algorithms")
    box(8.30, 1.60, 1.6, 1.85, "Friedman\n+ Nemenyi\n\nWilcoxon\n+ Holm\n\n$\\hat{A}_{12}$")
    for y in (4.07, 2.52, 0.97):
        arrow(2.32, y, 3.00, 3.05 if y > 3.5 else (2.20 if y < 1.5 else 2.60))
    arrow(5.17, 3.20, 5.90, 2.90)
    arrow(5.17, 1.65, 5.90, 2.10)
    arrow(7.88, 2.52, 8.25, 2.52)
    save(fig, "fig1_design")

### `fig2_cd`

Figure 2: critical-difference diagram with labels kept clear of leader lines.


In [ ]:
def fig2_cd(ranks, cd):
    """Critical difference diagram with labels held clear of the leader lines."""
    names, vals = list(ranks.index), ranks.values
    lo, hi = np.floor(vals.min()) - 0.5, np.ceil(vals.max()) + 0.5
    pad = 1.35                       # horizontal room reserved for text
    half = int(np.ceil(len(names) / 2))
    row_h = 0.34
    top = 0.0
    bottom = -row_h * (half + 1.6)

    fig, ax = plt.subplots(figsize=(7.0, 3.1))
    ax.set_xlim(hi + pad, lo - pad)          # reversed: rank 1 on the right
    ax.set_ylim(bottom, 1.05)
    ax.axis("off")

    ax.hlines(top, lo, hi, color="k", lw=1.1)
    for tck in np.arange(np.ceil(lo), np.floor(hi) + 1):
        ax.vlines(tck, top, top + 0.07, color="k", lw=1.0)
        ax.text(tck, top + 0.13, f"{int(tck)}", ha="center", va="bottom", fontsize=8)

    ax.plot([lo, lo - cd], [top + 0.62, top + 0.62], color="k", lw=3.0,
            solid_capstyle="butt")
    ax.text(lo - cd / 2, top + 0.68, f"critical difference = {cd:.2f}",
            ha="center", va="bottom", fontsize=8)

    for i, (nm, v) in enumerate(zip(names, vals)):
        left = i < half
        y = top - row_h * ((i if left else i - half) + 1)
        x_end = (lo - 0.10) if left else (hi + 0.10)
        ax.plot([v, v], [top, y], color="k", lw=0.7)
        ax.plot([v, x_end], [y, y], color="k", lw=0.7)
        ax.text(x_end - 0.06 if left else x_end + 0.06, y,
                f"{nm} ({v:.2f})", ha="right" if left else "left",
                va="center", fontsize=8)
    save(fig, "fig2_critical_difference")

### `fig3_wtl`

Figure 3: win/tie/loss heat map, metaphor-based algorithms vs baselines.


In [ ]:
def fig3_wtl(R):
    M = np.zeros((len(TEST_GROUP), len(BASELINES), 3))
    for i, a in enumerate(TEST_GROUP):
        for j, b in enumerate(BASELINES):
            s = R[(R.test == a) & (R.base == b)].outcome.value_counts()
            M[i, j] = [s.get("win", 0), s.get("tie", 0), s.get("loss", 0)]
    net = M[:, :, 0] - M[:, :, 2]

    fig, ax = plt.subplots(figsize=(5.8, 3.4))
    im = ax.imshow(net, cmap="Greys_r", vmin=-9, vmax=9)
    ax.set_xticks(range(len(BASELINES)), BASELINES)
    ax.set_yticks(range(len(TEST_GROUP)), TEST_GROUP)
    for i in range(len(TEST_GROUP)):
        for j in range(len(BASELINES)):
            w, t, l = M[i, j].astype(int)
            ax.text(j, i, f"{w}/{t}/{l}", ha="center", va="center", fontsize=8,
                    color="white" if net[i, j] < -2 else "black")
    ax.set_xlabel("baseline"); ax.set_ylabel("metaphor-based algorithm")
    ax.grid(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label("net wins (wins − losses)", fontsize=8)
    save(fig, "fig3_win_tie_loss")

### `fig4_ecdf`

Figure 4: anytime ECDF of runs within 1% of the published optimum.


In [ ]:
def fig4_ecdf(dm):
    fes = np.linspace(BUDGET / 100, BUDGET, 100)
    fig, ax = plt.subplots(figsize=(6.6, 3.6))
    for a in ORDER:
        sub = dm[dm.algorithm == a]
        T = np.array([np.array(x) for x in sub["trace"]])
        fs = sub["fstar"].values[:, None]
        gap = np.abs(T - fs) / np.abs(fs)
        frac = (np.where(np.isfinite(gap), gap, np.inf) <= 0.01).mean(axis=0)
        ax.plot(fes, frac, DASH[a], marker=MARK[a], markevery=14, ms=4, lw=1.2,
                color=COL[a], label=a)
    ax.set_xlabel("function evaluations")
    ax.set_ylabel("fraction of runs within 1% of published best")
    ax.set_ylim(-0.02, 1.02)
    ax.legend(ncol=1, loc="center left", bbox_to_anchor=(1.01, 0.5),
              frameon=False, handlelength=3.0)
    save(fig, "fig4_ecdf")

### `fig5_convergence`

Figure 5: median relative-gap convergence curves for four representative problems.


In [ ]:
def fig5_convergence(dm, problems):
    fes = np.linspace(BUDGET / 100, BUDGET, 100)
    fig, axes = plt.subplots(2, 2, figsize=(7.0, 4.8))
    for ax, pn in zip(axes.ravel(), problems):
        sub = dm[dm.problem == pn]
        fstar = PROBLEM_MAP[pn].known_f
        for a in ORDER:
            T = np.array([np.array(x) for x in sub[sub.algorithm == a]["trace"]])
            gap = np.abs(T - fstar) / abs(fstar)
            med = np.median(np.where(np.isfinite(gap), gap, 10.0), axis=0)
            ax.plot(fes, np.maximum(med, 1e-12), DASH[a], marker=MARK[a],
                    markevery=18, ms=3, lw=0.9, color=COL[a], label=a)
        ax.set_yscale("log"); ax.set_title(pn)
        ax.set_xlabel("function evaluations")
        ax.set_ylabel("median relative gap")
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.07),
               frameon=False, handlelength=3.0)
    fig.tight_layout()
    save(fig, "fig5_convergence")

### `fig6_box`

Figure 6: boxplot of relative gap to the published optimum, grouped by algorithm family.


In [ ]:
def fig6_box(dm, problems):
    fig, ax = plt.subplots(figsize=(7.0, 3.4))
    data = [np.maximum(dm[(dm.algorithm == a) & (dm.problem.isin(problems))]["gap"].values, 1e-9)
            for a in ORDER]
    bp = ax.boxplot(data, tick_labels=ORDER, showfliers=True, patch_artist=True,
                    widths=0.6, flierprops=dict(marker="+", ms=3, mew=0.6),
                    medianprops=dict(color="k", lw=1.2))
    for i, b in enumerate(bp["boxes"]):
        b.set_facecolor("0.92" if ORDER[i] in TEST_GROUP else "0.6")
        b.set_edgecolor("k"); b.set_linewidth(0.7)
    ax.axvline(len(TEST_GROUP) + 0.5, color="k", lw=0.8, ls=":")
    ax.text(3.5, ax.get_ylim()[1], "metaphor-based", ha="center", va="top", fontsize=8)
    ax.text(9.0, ax.get_ylim()[1], "baselines", ha="center", va="top", fontsize=8)
    ax.set_yscale("log"); ax.set_ylabel("relative gap to published best")
    ax.set_xticklabels(ORDER, rotation=45, ha="right")
    save(fig, "fig6_boxplot")

### `fig7_tuning`

Figure 7: effect of matched-budget tuning on held-out problems.


In [ ]:
def fig7_tuning(dm, dt, heldout):
    def mg(df):
        return (df[df.problem.isin(heldout)]
                .groupby(["algorithm", "problem"])["gap"].median()
                .unstack()[heldout].mean(axis=1))
    d0, d1 = mg(dm), mg(dt)
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    for a in ORDER:
        ax.plot([0, 1], [max(d0[a], 1e-6), max(d1[a], 1e-6)], DASH[a],
                marker=MARK[a], ms=5, lw=1.2, color=COL[a], label=a)
    ax.set_xticks([0, 1], ["author defaults", "matched-budget tuning"])
    ax.set_yscale("log"); ax.set_xlim(-0.08, 1.08)
    ax.set_ylabel("mean median relative gap (5 held-out problems)")
    ax.legend(ncol=1, loc="center left", bbox_to_anchor=(1.01, 0.5),
              frameon=False, handlelength=3.0)
    save(fig, "fig7_tuning")

### `fig8_constraint`

Figure 8: sensitivity to the three constraint-handling schemes.


In [ ]:
def fig8_constraint(dm, ds, de, problems):
    sets = {"Deb feasibility rules": dm[dm.problem.isin(problems)],
            "static penalty": ds, "$\\epsilon$-constrained": de}
    vals = {k: [max(v.groupby("algorithm")["gap"].mean().get(a, np.nan), 1e-8) for a in ORDER]
            for k, v in sets.items()}
    x = np.arange(len(ORDER)); w = 0.27
    hatch = ["", "///", "..."]; shade = ["0.30", "0.60", "0.85"]
    fig, ax = plt.subplots(figsize=(7.0, 3.5))
    for i, (k, v) in enumerate(vals.items()):
        ax.bar(x + (i - 1) * w, v, w, label=k, hatch=hatch[i],
               color=shade[i], edgecolor="k", linewidth=0.6)
    ax.set_yscale("log")
    ax.set_xticks(x, ORDER, rotation=45, ha="right")
    ax.set_ylabel("mean relative gap")
    ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, 1.16))
    save(fig, "fig8_constraint_scheme")

### `main`

Generates figures 1-8 in the numbered order they are referenced in the manuscript text.


In [ ]:
def main():
    dm, dt = load("main"), load("tuned")
    ds, de = load("cons", "static"), load("cons", "eps")
    allp = list(dm.problem.unique())
    scale = [p for p in allp if p != "GearTrain"]
    heldout = [p for p in scale if p not in TRAIN_PROBLEMS]
    consp = list(ds.problem.unique())
    _, _, _, ranks, cd = friedman_nemenyi(dm, allp)
    R = pairwise_wilcoxon(dm, allp)

    fig1_design()
    fig2_cd(ranks, cd)
    fig3_wtl(R)
    fig4_ecdf(dm)
    fig5_convergence(dm, ["WeldedBeam", "SpeedReducer", "PressureVessel", "TensionSpring"])
    fig6_box(dm, scale)
    fig7_tuning(dm, dt, heldout)
    fig8_constraint(dm, ds, de, consp)
    print("figures 1-8 written as PNG + PDF")

In [ ]:
if __name__ == "__main__":
    main()

---
## Key outcomes

- Produces the final, manuscript-numbered figure set (figs 1-8, PNG + PDF) directly from the `main`,
  `tuned` and `cons` result files; regenerating them reproduces the same statistics reported in
  notebook 06 (Friedman chi^2 = 52.50, p = 9.2e-8; Nemenyi CD = 5.03).
- Figure 2 (critical-difference diagram) visualises the same ranking as notebook 06: L-SHADE, DE and
  CMA-ES cluster at the top; HHO is a clear outlier at the bottom, separated from the rest by more than
  the critical difference.
- Figure 3 (win/tie/loss heat map) shows every metaphor-based algorithm losing more paired comparisons
  than it wins against every baseline, most severely HHO vs. L-SHADE/DE/CMA-ES.
- Figure 8 (constraint-scheme sensitivity) confirms the algorithm ranking is materially unchanged across
  the three constraint-handling schemes tested.

*Part of the budget-controlled reproduction study of nature-inspired metaheuristics.*


---

**Author:** Prakash Ukhalkar ([ORCID: 0000-0002-0452-6574](https://orcid.org/0000-0002-0452-6574)) — Pimpri Chinchwad College of Engineering, Pune, India

**Repository:** [github.com/prakash-ukhalkar/metaheuristic-budget-reproduction](https://github.com/prakash-ukhalkar/metaheuristic-budget-reproduction) &nbsp;|&nbsp; **License:** [MIT](../LICENSE) &nbsp;|&nbsp; **Citation:** [CITATION.cff](../CITATION.cff)

[![Repo](https://img.shields.io/badge/GitHub-metaheuristic--budget--reproduction-181717?logo=github&logoColor=white)](https://github.com/prakash-ukhalkar/metaheuristic-budget-reproduction) [![License](https://img.shields.io/badge/license-MIT-green)](../LICENSE)
